# 🌊 HormuzWatch — Maritime Anomaly Detection Model Benchmark
### Comprehensive Evaluation across Unsupervised, Semi-Supervised & Reconstruction Architectures

**Objective:** Identify the optimal ML anomaly detection model for maritime vessels and aviation tracking across the **Strait of Hormuz** chokepoint.

**Key Challenge:** High class imbalance ($\approx 3.2\%$ anomalies), non-stationary telemetry, and strict sub-millisecond inference latency SLAs.

In [ ]:
import os
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.metrics import (
    precision_recall_curve, auc, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix, classification_report
)
import joblib

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Environment initialized successfully.")

## 1. Locate and Load Dataset from `/server/datasets/`
The dataset contains vessel telemetry partitioned with grouped MMSI stratification to prevent temporal data leakage.

In [ ]:
# Locate dataset directory across local workspace and server mount paths
search_paths = [
    Path("/server/datasets"),
    Path("server/datasets"),
    Path("../server/datasets"),
    Path("../../server/datasets")
]

dataset_dir = None
for p in search_paths:
    if p.exists():
        vessel_dirs = list(p.glob("dataset_vessel_*"))
        if vessel_dirs:
            dataset_dir = sorted(vessel_dirs)[-1]
            break

if dataset_dir is None:
    raise FileNotFoundError("No dataset_vessel_* folder found in candidate search paths.")

print(f"Selected Dataset: {dataset_dir}")

# Load metadata
with open(dataset_dir / "metadata.json", "r") as f:
    metadata = json.load(f)
print("Dataset Metadata:", json.dumps(metadata, indent=2))

# Load Train, Validation, and Test splits
train_df = pd.read_csv(dataset_dir / "train.csv")
val_df = pd.read_csv(dataset_dir / "val.csv")
test_df = pd.read_csv(dataset_dir / "test.csv")

feature_cols = metadata["feature_columns"]
print(f"\nFeatures ({len(feature_cols)}): {feature_cols}")
print(f"Train samples: {len(train_df):,}")
print(f"Val samples:   {len(val_df):,}")
print(f"Test samples:  {len(test_df):,}")

## 2. Exploratory Data Analysis & Class Balance

In [ ]:
# Compute ground truth anomaly binary flag (1 = anomaly, 0 = normal)
train_labels = (train_df["label"] == "anomaly").astype(int)
val_labels = (val_df["label"] == "anomaly").astype(int)
test_labels = (test_df["label"] == "anomaly").astype(int)

print("Anomaly Rates:")
print(f"  Train: {train_labels.mean()*100:.2f}%")
print(f"  Val:   {val_labels.mean()*100:.2f}%")
print(f"  Test:  {test_labels.mean()*100:.2f}%")

# Feature correlation heatmap
corr = train_df[feature_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Kinematic Feature Correlation Matrix (Hormuz Chokepoint Telemetry)")
plt.show()

## 3. Preprocessing & Feature Standardization
Fit a `StandardScaler` on the training distribution and transform all partitions.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols].fillna(0.0))
X_val = scaler.transform(val_df[feature_cols].fillna(0.0))
X_test = scaler.transform(test_df[feature_cols].fillna(0.0))

y_train = train_labels.values
y_val = val_labels.values
y_test = test_labels.values
print(f"Processed feature matrix shape: {X_train.shape}")

## 4. Multi-Model Benchmark Suite
We train and benchmark 6 models:
1. **Isolation Forest** (Fast path length isolation)
2. **Local Outlier Factor** (Local density novelty detector)
3. **One-Class SVM** (Non-linear RBF kernel boundary)
4. **Elliptic Envelope** (FastMCD Mahalanobis distance)
5. **PCA/SVD Reconstruction Autoencoder** (Linear bottleneck reconstruction error)
6. **Supervised Random Forest Baseline** (Class-weighted ensemble)

In [ ]:
models = {}
results = []

# 1. Isolation Forest
print("Training Isolation Forest...")
t0 = time.time()
iforest = IsolationForest(n_estimators=150, contamination=0.035, max_samples=0.8, random_state=42, n_jobs=-1)
iforest.fit(X_train)
train_time_if = time.time() - t0
models["Isolation Forest"] = iforest

# 2. Local Outlier Factor (Subsampled fit for density estimation)
print("Training Local Outlier Factor...")
t0 = time.time()
lof = LocalOutlierFactor(n_neighbors=35, contamination=0.035, novelty=True, n_jobs=-1)
lof.fit(X_train[:30000]) # Subsample 30k for tractable neighbor graph
train_time_lof = time.time() - t0
models["Local Outlier Factor"] = lof

# 3. One-Class SVM (Subsampled fit)
print("Training One-Class SVM...")
t0 = time.time()
ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.035)
ocsvm.fit(X_train[:20000]) # Subsample for quadratic kernel scaling
train_time_ocsvm = time.time() - t0
models["One-Class SVM"] = ocsvm

# 4. Elliptic Envelope (Mahalanobis distance)
print("Training Elliptic Envelope...")
t0 = time.time()
envelope = EllipticEnvelope(contamination=0.035, random_state=42)
envelope.fit(X_train[:40000])
train_time_env = time.time() - t0
models["Elliptic Envelope"] = envelope

# 5. Reconstruction Autoencoder (PCA Bottleneck)
print("Fitting Reconstruction Autoencoder...")
t0 = time.time()
U, S, Vt = np.linalg.svd(X_train - np.mean(X_train, axis=0), full_matrices=False)
latent_k = 4
W_enc = Vt[:latent_k, :].T
train_time_ae = time.time() - t0
models["Reconstruction Autoencoder"] = {"W_enc": W_enc, "mean": np.mean(X_train, axis=0)}

# 6. Supervised Random Forest Baseline
print("Training Random Forest Baseline...")
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
train_time_rf = time.time() - t0
models["Random Forest (Supervised)"] = rf

print("All candidate models trained successfully!")

## 5. Model Inference, Scoring & Latency Profiling
Compute anomaly score vectors and measure single-sample CPU latency.

In [ ]:
eval_records = []
scores_dict = {}

# 1. Isolation Forest Scoring
raw_if = -iforest.score_samples(X_test)
scores_dict["Isolation Forest"] = raw_if

# Latency benchmark (single sample)
t_lat = time.perf_counter()
for _ in range(500):
    _ = iforest.score_samples(X_test[:1])
lat_if = ((time.perf_counter() - t_lat) / 500) * 1000.0

# 2. LOF Scoring
raw_lof = -lof.score_samples(X_test)
scores_dict["Local Outlier Factor"] = raw_lof
t_lat = time.perf_counter()
for _ in range(500):
    _ = lof.score_samples(X_test[:1])
lat_lof = ((time.perf_counter() - t_lat) / 500) * 1000.0

# 3. One-Class SVM Scoring
raw_ocsvm = -ocsvm.decision_function(X_test)
scores_dict["One-Class SVM"] = raw_ocsvm
t_lat = time.perf_counter()
for _ in range(500):
    _ = ocsvm.decision_function(X_test[:1])
lat_ocsvm = ((time.perf_counter() - t_lat) / 500) * 1000.0

# 4. Elliptic Envelope Scoring
raw_env = envelope.mahalanobis(X_test)
scores_dict["Elliptic Envelope"] = raw_env
t_lat = time.perf_counter()
for _ in range(500):
    _ = envelope.mahalanobis(X_test[:1])
lat_env = ((time.perf_counter() - t_lat) / 500) * 1000.0

# 5. Autoencoder Reconstruction Loss
ae = models["Reconstruction Autoencoder"]
centered = X_test - ae["mean"]
recon = np.dot(np.dot(centered, ae["W_enc"]), ae["W_enc"].T) + ae["mean"]
raw_ae = np.mean((X_test - recon) ** 2, axis=1)
scores_dict["Reconstruction Autoencoder"] = raw_ae
t_lat = time.perf_counter()
for _ in range(500):
    _c = X_test[:1] - ae["mean"]
    _ = np.mean((_c - np.dot(np.dot(_c, ae["W_enc"]), ae["W_enc"].T)) ** 2)
lat_ae = ((time.perf_counter() - t_lat) / 500) * 1000.0

# 6. Random Forest Supervised Probability
raw_rf = rf.predict_proba(X_test)[:, 1]
scores_dict["Random Forest (Supervised)"] = raw_rf
t_lat = time.perf_counter()
for _ in range(500):
    _ = rf.predict_proba(X_test[:1])
lat_rf = ((time.perf_counter() - t_lat) / 500) * 1000.0

latencies = {
    "Isolation Forest": lat_if,
    "Local Outlier Factor": lat_lof,
    "One-Class SVM": lat_ocsvm,
    "Elliptic Envelope": lat_env,
    "Reconstruction Autoencoder": lat_ae,
    "Random Forest (Supervised)": lat_rf,
}

## 6. Precision-Recall AUC & Performance Benchmarks

In [ ]:
plt.figure(figsize=(10, 7))

benchmark_summary = []
for name, scores in scores_dict.items():
    precision, recall, thresholds = precision_recall_curve(y_test, scores)
    pr_auc = auc(recall, precision)
    roc_auc = roc_auc_score(y_test, scores)
    
    # Optimal F1 threshold
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_thresh = thresholds[min(best_idx, len(thresholds)-1)]
    
    # Top-50 alert precision
    top50_idx = np.argsort(scores)[-50:]
    p_at_50 = y_test[top50_idx].mean() * 100.0
    
    benchmark_summary.append({
        "Model": name,
        "PR-AUC": round(pr_auc, 4),
        "ROC-AUC": round(roc_auc, 4),
        "Best F1": round(best_f1, 4),
        "Precision @ 50": f"{p_at_50:.1f}%",
        "Latency (ms)": round(latencies[name], 3),
    })
    
    plt.plot(recall, precision, label=f"{name} (PR-AUC = {pr_auc:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves on Holdout Test Set (Hormuz Telemetry)")
plt.legend(loc="lower left")
plt.show()

df_benchmark = pd.DataFrame(benchmark_summary).sort_values("PR-AUC", ascending=False)
print("\n=================== MODEL BENCHMARK RANKING ===================")
print(df_benchmark.to_string(index=False))

## 7. Operational Recommendations & Production Model Bundle Export
Export winning model configuration for the pluggable `service/ml-service/core/` registry.

In [ ]:
winning_model_name = df_benchmark.iloc[0]["Model"]
print(f"Best Fit Model Selected: {winning_model_name}")

# Export artifacts
out_dir = Path("models_benchmark_export")
out_dir.mkdir(exist_ok=True)

joblib.dump(scaler, out_dir / "feature_scaler.joblib")
joblib.dump(iforest, out_dir / "vessel_iforest.joblib")

summary_report = {
    "benchmark_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "evaluated_models": len(scores_dict),
    "dataset": str(dataset_dir.name),
    "test_rows": len(test_df),
    "ranking": df_benchmark.to_dict(orient="records"),
    "winning_model": winning_model_name,
}

with open(out_dir / "benchmark_report.json", "w") as f:
    json.dump(summary_report, f, indent=2)

print(f"Exported benchmark report to {out_dir / 'benchmark_report.json'}")